# Day 4 — Prompt Design & Context Window Management

---

Retrieval brings back chunks. **How you feed those chunks to the LLM matters as much as what you retrieved.**

Today you'll learn:

1. A **RAG prompt template** you can reuse everywhere
2. **Context stuffing strategies** — top-k, character budgets, deduplication
3. **Inline citations** the model actually uses correctly
4. What to do when the context doesn't fit


## 1. The reusable RAG prompt template

Every professional RAG prompt has the same skeleton:

```
[SYSTEM]  You are a helpful assistant. Answer using ONLY the context.
          If the answer isn't in the context, say "I don't know."
          Cite sources with [1], [2], etc.

[CONTEXT] [1] <chunk 1>
          [2] <chunk 2>
          [3] <chunk 3>

[USER]    <question>
```

The three key phrases — `only the context`, `I don't know`, `cite with brackets` — are what separate a demo from a production RAG.


In [9]:
def build_prompt(question: str, chunks: list[str]) -> list[dict]:
    numbered = "\n\n".join(f"[{i+1}] {c}" for i, c in enumerate(chunks))
    system = (
        "You are a helpful assistant. "
        "Answer the user's question using ONLY the numbered context below. "
        "If the answer is not in the context, say 'I don't know.' "
        "Cite the sources you used with bracket numbers like [1], [2]."
    )
    user = f"Context:\n{numbered}\n\nQuestion: {question}"
    return [
        {"role": "system", "content": system},
        {"role": "user",   "content": user},
    ]

msgs = build_prompt("How much is the Pro plan?", [
    "The Pro plan costs $29/month and includes 500 GB.",
    "AcmeCloud servers are located in AWS us-east-1.",
])
for m in msgs:
    print(f"[{m['role']}]\n{m['content']}\n")


[system]
You are a helpful assistant. Answer the user's question using ONLY the numbered context below. If the answer is not in the context, say 'I don't know.' Cite the sources you used with bracket numbers like [1], [2].

[user]
Context:
[1] The Pro plan costs $29/month and includes 500 GB.

[2] AcmeCloud servers are located in AWS us-east-1.

Question: How much is the Pro plan?



## 2. Context stuffing — how much do I put in?

You can't just dump 100 chunks into the prompt. Three limits force restraint:

1. **The context window** — every LLM has a max (Llama-3.3-70B: 128k tokens; older models: 4k–32k).
2. **Cost** — you pay per token *in* and *out*. Bigger prompts = bigger bills.
3. **Quality** — LLMs get **worse** with irrelevant context. This is called *distraction*.

**Practical rules:**

- Top-k between **3 and 6** for most apps.
- Cap total context at **~2000 characters** (~500 tokens). Enough to answer; small enough to stay sharp and cheap.
- **Deduplicate** — if two retrieved chunks are near-identical, keep one.


In [10]:
def dedupe(chunks: list[str], min_diff: int = 30) -> list[str]:
    """Drop a chunk if it starts with the same 30 chars as an earlier one."""
    seen, out = set(), []
    for c in chunks:
        key = c[:min_diff].strip().lower()
        if key not in seen:
            seen.add(key)
            out.append(c)
    return out


def cap_context(chunks: list[str], max_chars: int = 2000) -> list[str]:
    """Take chunks in order until we hit max_chars total."""
    total, out = 0, []
    for c in chunks:
        if total + len(c) > max_chars:
            break
        out.append(c)
        total += len(c)
    return out


raw = [
    "The Pro plan costs $29/month.",
    "The Pro plan costs $29 monthly and includes 500 GB.",   # near-duplicate
    "Enterprise customers get 24/7 support.",
    "AcmeCloud was founded in 2019 by Priya Rao and Marcus Chen.",
]
print("After dedupe:", dedupe(raw))
print("After cap :", cap_context(dedupe(raw), max_chars=80))


After dedupe: ['The Pro plan costs $29/month.', 'The Pro plan costs $29 monthly and includes 500 GB.', 'Enterprise customers get 24/7 support.', 'AcmeCloud was founded in 2019 by Priya Rao and Marcus Chen.']
After cap : ['The Pro plan costs $29/month.', 'The Pro plan costs $29 monthly and includes 500 GB.']


## 3. Counting tokens properly

Character counts are a rough proxy for tokens. If you need exact numbers (e.g. to enforce a hard limit), use `tiktoken`:


In [ ]:
!pip install tiktoken --quiet

In [3]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o-mini")  # close enough for other models too

def n_tokens(text: str) -> int:
    return len(enc.encode(text))

def cap_tokens(chunks: list[str], max_tokens: int = 500) -> list[str]:
    total, out = 0, []
    for c in chunks:
        t = n_tokens(c)
        if total + t > max_tokens:
            break
        out.append(c)
        total += t
    return out

print(n_tokens("This is a short test sentence."))


7


## 4. Inline citations — get the model to actually use them

You told the model to cite with `[1]`, `[2]`. That works ~80% of the time out of the box. Two tricks boost it to ~99%:

1. **Number the chunks in the prompt** (we already did that).
2. **Add a "Format your answer like this:" example** at the end of the system prompt.


In [4]:
def build_prompt_v2(question: str, chunks: list[str]) -> list[dict]:
    numbered = "\n\n".join(f"[{i+1}] {c}" for i, c in enumerate(chunks))
    system = (
        "You are a helpful assistant. Answer using ONLY the numbered context. "
        "If the answer is not there, say 'I don't know.' "
        "Cite sources with bracket numbers.\n\n"
        "Format your answer like this:\n"
        "The Pro plan costs $29/month [2]."
    )
    return [
        {"role": "system", "content": system},
        {"role": "user",   "content": f"Context:\n{numbered}\n\nQuestion: {question}"},
    ]


**Why one example beats a paragraph of instructions:** LLMs are pattern-matchers. Show them the pattern once and they'll follow it. This trick is called a **one-shot example** and is stolen straight from Section 4 Day 4.


## 5. Putting it all together


In [8]:
import os
from dotenv import load_dotenv
from together import Together
load_dotenv()
llm = Together()

def full_rag(question: str, raw_chunks: list[str]) -> str:
    chunks = cap_tokens(dedupe(raw_chunks), max_tokens=500)
    messages = build_prompt_v2(question, chunks)
    resp = llm.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=messages,
        temperature=0.2,
    )
    return resp.choices[0].message.content

# Simulate having retrieved these chunks
retrieved = [
    "The free tier includes 10 GB of storage.",
    "The Pro plan costs $29/month and includes 500 GB.",
    "Enterprise customers receive 24/7 phone support.",
]
print(full_rag("tell me about the pricing", retrieved))


The free tier includes 10 GB of storage [1]. The Pro plan costs $29/month and includes 500 GB [2].


**Expected output:** something like `"The free tier is $0 with 10 GB [1], while the Pro plan is $29/month with 500 GB [2]."`

Notice the bracketed citations. That's what you want.


## 6. When the context doesn't fit

Say the user asks a broad question that requires 20 relevant chunks — but you can only fit 5. Three options:

1. **Summarize first** — ask the LLM to summarize each chunk before packing them.
2. **Map-reduce** — answer with each chunk separately, then combine the mini-answers.
3. **Recursive retrieval** — answer using top-5, notice you need more, retrieve again with a refined query.

For freshers: option **1** is easiest — summarize each chunk in ~2 sentences before stuffing. The other patterns belong in Section 7 (Agents).


## Recap

- Use the **system + numbered context + question** template. Every time.
- **Dedupe** and **cap context** — 3–6 chunks, ~500 tokens.
- **Number chunks** and show a **one-shot example** to make citations reliable.
- Use `tiktoken` when you need exact token counts.
- When context doesn't fit: **summarize each chunk** before stuffing.
- **Next class:** hallucinations, prompt injection, and how to measure quality.
